# Ollama LLM & Embeddings With llama-index-pydocker

Running Ollama locally requires pulling a Docker image, waiting for the API to
be ready, and then downloading whichever model you need before any inference
can happen.

This notebook shows how `llama-index-pydocker` handles all of that
automatically â€” one import swap starts the container and streams model
download progress via tqdm.  The same Docker container is shared by both
the embedding model and the LLM.

## Table Of Contents

- [Prerequisites](#prerequisites)
- [1. Start Ollama With One Import Swap](#1-start-ollama-with-one-import-swap)
- [2. Embed Text With OllamaEmbedding](#2-embed-text-with-ollamaembedding)
- [3. Generate Text With the Ollama LLM](#3-generate-text-with-the-ollama-llm)
- [4. Context Manager Teardown](#4-context-manager-teardown)
- [5. Remote Passthrough (No Docker)](#5-remote-passthrough-no-docker)
- [6. Conclusion](#6-conclusion)

## Prerequisites

- Docker Desktop must be running.
- Install dependencies:
  ```bash
  pip install "llama-index-pydocker[ollama]"
  ```
- No environment variables required.
- First run pulls the Ollama image and the model weights â€” allow a few minutes
  depending on your connection speed.

## 1. Start Ollama With One Import Swap

Change the import from `llama_index.llms.ollama` / `llama_index.embeddings.ollama`
to `llama_index_pydocker`.  The Ollama container starts automatically when
`base_url` resolves to localhost.

A single `OllamaConfig` with a fixed `container_name` is shared by both the
embedding model and the LLM so they reuse the same container.

In [1]:
import sys
import uuid
import tempfile
from pathlib import Path

sys.path.insert(0, str(Path().cwd().parent))

from llama_index_pydocker import Ollama, OllamaEmbedding
from docker_db import OllamaConfig

In [2]:
temp_dir = Path(tempfile.mkdtemp())
container_name = f"demo-ollama-{uuid.uuid4().hex[:8]}"

# Shared config â€” both embedding and LLM will reference this container.
# The second instantiation finds it already running and skips startup.
cfg = OllamaConfig(
    project_name="demo",
    container_name=container_name,
    volume_path=temp_dir / "ollamadata",
    retries=30,
    delay=2,
)

print(f"Container name:  {container_name}")
print(f"Ollama port:     {cfg.port}")

Container name:  demo-ollama-4ba03df3
Ollama port:     11434


## 2. Embed Text With OllamaEmbedding

`OllamaEmbedding` starts the container, then checks whether `all-minilm`
is already present.  If not, it streams the download with one tqdm progress
bar per layer so you always know what you're waiting for.

`all-minilm` is only 46 MB â€” the smallest embedding model available on Ollama.

In [3]:
EMBED_MODEL = "all-minilm"
base_url = f"http://localhost:{cfg.port}"

embed_model = OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=base_url,
    docker_config=cfg,
)

print(f"OllamaEmbedding ready.  Container: {embed_model._db.config.container_name}")

Pulling Ollama model 'all-minilm' ...


  pulling manifest


  pulling 797b70c4edf8 sha256:797b7:   0%|          | 0.00/43.8M [00:00<?, ?B/s]

  pulling 797b70c4edf8 sha256:797b7:   0%|          | 0.00/43.8M [00:00<?, ?B/s]

  pulling 797b70c4edf8 sha256:797b7:   0%|          | 0.00/43.8M [00:00<?, ?B/s]

...

  pulling 548455b72658 sha256:54845: 100%|██████████| 407/407 [00:00<00:00, 470kB/s] 


  verifying sha256 digest


  writing manifest


  success


OllamaEmbedding ready.  Container: demo-ollama-4ba03df3


In [4]:
texts = [
    "Ollama runs large language models locally.",
    "Docker makes local infrastructure reproducible for development and testing.",
    "all-minilm is a compact embedding model optimised for retrieval tasks.",
]

embeddings = embed_model.get_text_embedding_batch(texts)

print(f"Embedded {len(embeddings)} texts.")
print(f"Embedding dimension: {len(embeddings[0])}")

assert len(embeddings) == len(texts)
assert all(len(e) > 0 for e in embeddings)

Embedded 3 texts.
Embedding dimension: 384


## 3. Generate Text With the Ollama LLM

`Ollama` reuses the same container (same `container_name`).  `tinyllama` is
pulled if not already present, then a simple completion is run to confirm
end-to-end inference is working.

In [5]:
LLM_MODEL = "tinyllama"

llm = Ollama(
    model=LLM_MODEL,
    base_url=base_url,
    docker_config=cfg,
    request_timeout=120.0,
)

print(f"Ollama LLM ready.  Container: {llm._db.config.container_name}")

Container demo-ollama-4ba03df3 already exists.


Pulling Ollama model 'tinyllama' ...


  pulling manifest


  pulling 2af3b81862c6 sha256:2af3b:   0%|          | 0.00/608M [00:00<?, ?B/s]

  pulling 2af3b81862c6 sha256:2af3b:   0%|          | 0.00/608M [00:00<?, ?B/s]

...

  pulling 6331358be52a sha256:63313: 100%|██████████| 483/483 [00:00<00:00, 377kB/s] 


  verifying sha256 digest


  writing manifest


  success


Ollama LLM ready.  Container: demo-ollama-4ba03df3


In [6]:
response = llm.complete("In one sentence, what is Docker?")

print(f"Response: {response.text.strip()}")

assert len(response.text.strip()) > 0

Response: Docker is a platform that enables organizations to build, ship, and run distributed applications across multiple environments. It helps you manage your infrastructure more efficiently and easily, while also enabling faster development cycles.


## 4. Context Manager Teardown

Call `stop()` explicitly or use `with` so teardown happens automatically.
Note: stopping via `embed_model` or `llm` both stop the **same** container â€”
only call `stop()` once.

In [7]:
embed_model.stop()
print(f"Container '{container_name}' removed.")

Container demo-ollama-4ba03df3 stopped and port 11434 is free


Container 'demo-ollama-4ba03df3' removed.


In [8]:
# Context manager pattern â€” container is removed on exit
new_name = f"demo-ollama-{uuid.uuid4().hex[:8]}"
new_cfg = cfg.model_copy(update={"container_name": new_name,
                                   "volume_path": temp_dir / new_name})

with OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=base_url,
    docker_config=new_cfg,
) as em:
    print(f"Container up: {em._db.config.container_name}")

print("Container removed.")

Pulling Ollama model 'all-minilm' ...


  pulling manifest


  pulling 797b70c4edf8 sha256:797b7:   0%|          | 0.00/43.8M [00:00<?, ?B/s]

  pulling 797b70c4edf8 sha256:797b7:   0%|          | 0.00/43.8M [00:00<?, ?B/s]

  pulling 797b70c4edf8 sha256:797b7:   0%|          | 0.00/43.8M [00:00<?, ?B/s]

...

  writing manifest


  success


Container up: demo-ollama-b1e05804


Container demo-ollama-b1e05804 stopped and port 11434 is free


Container removed.


## 5. Remote Passthrough (No Docker)

Point either class at a remote Ollama server and the wrapper is completely
transparent â€” Docker is never started.

In [9]:
# Uncomment to target a remote Ollama deployment â€” Docker is never started
#
# embed_model = OllamaEmbedding(
#     model_name="nomic-embed-text",
#     base_url="http://my-gpu-server:11434",
# )
# assert embed_model._db is None
#
# llm = Ollama(
#     model="llama3",
#     base_url="http://my-gpu-server:11434",
# )
# assert llm._db is None

print("Remote passthrough: Docker is never touched for non-localhost URLs.")

Remote passthrough: Docker is never touched for non-localhost URLs.


## 6. Conclusion

You have completed the workflow introduced at the top:
1. Started an Ollama container with a single import swap.
2. Pulled `all-minilm` (46 MB) with live per-layer tqdm progress bars.
3. Embedded a batch of texts and confirmed the embedding dimension.
4. Pulled `tinyllama`, ran a completion, and confirmed the response.
5. Cleaned up with `stop()` and with a context manager.

Key next steps:
- Swap `all-minilm` for any Ollama-hosted embedding model and update
  your vector store's `embed_dim` to match.
- Swap `tinyllama` for a larger model such as `llama3` or `mistral` when you
  need better generation quality.
- Use `OllamaConfig.volume_path` to persist downloaded model weights across
  runs so the pull only happens once.
- Switch `base_url` to a remote host to deploy without any other code changes.